# 001 深度学习问题与证据

先完成纸笔任务。程序仅核对原创合成数据；测试标签公开，不是未知盲测。002会逐步讲解语法。

In [1]:
from pathlib import Path
from experiment import read_rows, choose_bias, lookup_predictions, align_labels, mae, run_self_tests, run
base = Path.cwd()
training = read_rows(base / "data/train.csv", ["id", "x", "y"])
inputs = read_rows(base / "data/test_inputs.csv", ["id", "x"])
print("训练条数", len(training), "测试输入条数", len(inputs))

训练条数 5 测试输入条数 4


## 用训练资料选偏移量
预期候选0、1、2的平均绝对误差为1.2、0.2、0.8。

In [2]:
bias, candidates = choose_bias(training)
print(candidates)
print("所选偏移量", bias)
assert bias == 1

[{'bias': 0.0, 'train_mae': 1.2}, {'bias': 1.0, 'train_mae': 0.2}, {'bias': 2.0, 'train_mae': 0.8}]
所选偏移量 1.0


## 在揭示标签之前保存预测
A预测4、6、8、10，B预测0、0、0、0。此后不根据答案修改规则。

In [3]:
a_predictions = [2 * row["x"] + bias for row in inputs]
b_predictions = lookup_predictions(training, inputs)
print("A", a_predictions)
print("B", b_predictions)
assert a_predictions == [4, 6, 8, 10]
assert b_predictions == [0, 0, 0, 0]

A [4.0, 6.0, 8.0, 10.0]
B [0, 0, 0, 0]


## 揭示公开教学标签并核对
注意这里的文件分离不提供真正访问隔离。

In [4]:
labels = read_rows(base / "data/test_labels.csv", ["id", "y"])
actual = align_labels(inputs, labels)
print("A测试MAE", mae(actual, a_predictions))
print("B测试MAE", mae(actual, b_predictions))
assert mae(actual, a_predictions) == 0
assert mae(actual, b_predictions) == 7

A测试MAE 0.0
B测试MAE 7.0


## 只改变关系的压力测试
仍使用原预测，预期A误差3。

In [5]:
stress = align_labels(inputs, read_rows(base / "data/stress_labels.csv", ["id", "y"]))
print("A压力测试MAE", mae(stress, a_predictions))
assert mae(stress, a_predictions) == 3
print(run_self_tests())

A压力测试MAE 3.0
{'status': 'passed', 'checks': 11}


## 保存完整记录并复核手算
保存到notebook_outputs，不覆盖原始数据。最后重启内核并运行全部。

In [6]:
metrics = run(base / "notebook_outputs")
expected = {"selected_bias": 1, "A_train_mae": 0.2, "B_train_mae": 0, "A_test_mae": 0, "B_test_mae": 7, "A_stress_mae": 3}
for key, value in expected.items():
    assert abs(metrics[key] - value) < 1e-12
print(metrics)

{'selected_bias': 1.0, 'A_train_mae': 0.2, 'B_train_mae': 0.0, 'A_test_mae': 0.0, 'B_test_mae': 7.0, 'A_stress_mae': 3.0, 'n_train': 5, 'n_test': 4}


## 写结论
解释为什么B训练零误差不足以证明新输入表现。区分公开流程复现、合成压力测试与真实装置研究。